# ProcGen RL - training notebook

Trains the PPO agent and exports **ghost** files (one per training milestone: 0min, 30min, 1h, 2h, 5h, 10h, ...)
that the pygame demo on your laptop loads.

**Runtime:** a plain CPU runtime is enough (Runtime > Change runtime type > CPU). The policy is a small MLP, so the
time goes into simulating the game, not the network. A GPU does not make this faster and wastes your GPU quota.

**Safe to disconnect:** everything is written to Google Drive. If Colab disconnects, run all cells again and
training resumes from `latest.zip`.

Steps: 1) settings, 2) get the code, 3) install, 4) sanity check, 5) train, 6) watch TensorBoard, 7) evaluate, 8) download ghosts.

## 1. Settings

In [ ]:
HOURS = 100            # total simulated hours to train (100 sim h = 5.4M agent steps)
N_ENVS = 8             # parallel game copies
MAX_WALL_MINUTES = 0   # stop cleanly after this many real minutes (0 = no limit)
RUN_NAME = "main"      # change to start a separate run, e.g. "fixed10" for the generalisation experiment
EXTRA_ARGS = ""        # e.g. "--train-levels 10" for the generalisation experiment
USE_DRIVE = True       # save runs to Google Drive so they survive disconnects
REPO_URL = "https://github.com/DYNAMAXD/ProcGen_RL.git"

## 2. Storage and code
If you have not pushed the code yet, set `REPO_URL = None` and upload `ProcGen_RL.zip` (zip of the repo folder) when asked.

In [ ]:
import os, subprocess, sys
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    RUNS_ROOT = '/content/drive/MyDrive/procgen_rl_runs'
else:
    RUNS_ROOT = '/content/runs'
os.makedirs(RUNS_ROOT, exist_ok=True)
RUN_DIR = f'{RUNS_ROOT}/{RUN_NAME}'

CODE_DIR = '/content/ProcGen_RL'
if REPO_URL:
    if os.path.isdir(CODE_DIR):
        subprocess.run(['git', '-C', CODE_DIR, 'pull'], check=False)
    else:
        subprocess.run(['git', 'clone', REPO_URL, CODE_DIR], check=True)
else:
    from google.colab import files
    up = files.upload()
    name = next(iter(up))
    subprocess.run(['unzip', '-oq', name, '-d', '/content/'], check=True)
    if not os.path.isdir(CODE_DIR):
        # the zip may contain the folder under another name
        cand = [d for d in os.listdir('/content') if os.path.isfile(f'/content/{d}/train.py')]
        CODE_DIR = f'/content/{cand[0]}'
os.chdir(CODE_DIR)
print('code:', CODE_DIR, '| run dir:', RUN_DIR, '| cpus:', os.cpu_count())

## 3. Install

In [ ]:
!pip install -q -r requirements.txt

## 4. Sanity check
Runs the env tests and measures raw game speed.

In [ ]:
!python -m pytest -q tests/test_env.py
!python -m procgen_rl.bench

## 5. Train
Re-running this cell resumes from `RUN_DIR/latest.zip`. Ghosts appear in `RUN_DIR/ghosts` as soon as each milestone
is reached, so you can download partial results at any time.

In [ ]:
!python train.py --hours {HOURS} --n-envs {N_ENVS} --out "{RUN_DIR}" --max-wall-minutes {MAX_WALL_MINUTES} {EXTRA_ARGS} 2>&1 | grep -v -E "oneDNN|absl|cuda"

## 6. TensorBoard
Useful curves: `rollout/success_rate`, `rollout/progress`, `curriculum/difficulty`, `sim/hours`.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "{RUN_DIR}/tb"

## 7. Evaluate every ghost on unseen levels

In [ ]:
!python evaluate.py --ghosts "{RUN_DIR}/ghosts" --levels 100 --plot "{RUN_DIR}/eval.png"
from IPython.display import Image, display
display(Image(f'{RUN_DIR}/eval.png'))

## 8. Download the ghosts
Unzip into the `ghosts/` folder of the repo on your laptop, then run `python demo.py`.

In [ ]:
import shutil
zip_path = shutil.make_archive(f'/content/ghosts_{RUN_NAME}', 'zip', f'{RUN_DIR}/ghosts')
from google.colab import files
files.download(zip_path)